# 12 — Tuning del HistGradientBoosting (Paso 6)

Hasta acá todos los experimentos usaron el **mismo modelo fijo** (`learning_rate=0.1`, `max_iter=200`, el resto por defecto) para que la única diferencia entre bloques fueran las features. Ahora que las features están definidas, ajusto los hiperparámetros del modelo.

**Features = configuración ganadora de `11b_marginales.ipynb`**: T2a + N (fotos) + L (fechas) + T3 (`warranty` en el modelo de texto) = **0,9339 ± 0,0018** con el modelo fijo.

## Los hiperparámetros

Material: `material/modelos de ensamble/S8_Ensambles.ipynb` (Gradient Boosting: $F_m(x) = F_{m-1}(x) + \eta\, h_m(x)$; `learning_rate` y `n_estimators` "el equilibrio debe evaluarse con validación") y `S7_Árboles_de_decisión.ipynb` (`max_leaf_nodes`, `min_samples_leaf`).

| Hiperparámetro | Qué controla | Valores | Fuente |
|---|---|---|---|
| `learning_rate` ($\eta$) | cuánto pesa cada árbol nuevo; más chico = pasos más cortos, más árboles, suele generalizar mejor | 0,03 · 0,05 · 0,1 · 0,2 | S8 |
| `max_iter` | cantidad máxima de árboles (= `n_estimators`) | fijo en 2.000, lo decide el *early stopping* | S8 |
| `max_leaf_nodes` | hojas máximas de cada árbol = su complejidad | 15 · 31 · 63 · 127 | S7 |
| `min_samples_leaf` | mínimo de publicaciones por hoja; más alto = no memoriza casos sueltos | 10 · 20 · 50 · 100 | S7 |
| `l2_regularization` | penaliza valores grandes en las hojas (como Ridge) | 0 · 0,1 · 1 · 10 | **no está en clase**; idea de L2 en `Continuación de regresion y Regulización L1 y L2.pdf` |

**Early stopping (no está en el material).** El HistGB separa al azar el 10% **de su propio train** (`validation_fraction=0.1`), y después de cada árbol mira la pérdida ahí. Si en 10 iteraciones seguidas (`n_iter_no_change=10`) no mejora, para. Ese 10% sale del train del fold, nunca del fold de validación de la CV. Con un tope alto (`max_iter=2000`) el modelo elige solo cuántos árboles usar, y el equilibrio `learning_rate` ↔ cantidad de árboles se resuelve automáticamente. **Ojo:** el modelo fijo ya lo usaba sin que lo pidiéramos (`early_stopping="auto"` se prende con más de 10.000 filas) y paraba en ~110–120 árboles, así que el tope de 200 nunca importó.

**`RandomizedSearchCV` (no está en el material; en clase se usa `GridSearchCV`).** La grilla completa son 4⁴ = 256 combinaciones × 5 folds, y cada ajuste tarda. `RandomizedSearchCV` elige **30 combinaciones al azar** de esa misma grilla (`random_state=42`) y las evalúa con la misma CV. Con pocos hiperparámetros, unas decenas de combinaciones al azar suelen acercarse a la mejor de la grilla.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings
import tempfile
import shutil
from joblib import Memory

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder, PrediccionOOF

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

## Features de la configuración ganadora (N, L, T3)

Mismo código que en `11b_marginales.ipynb`. Son determinísticas, por eso van afuera del `Pipeline`.

In [2]:
# T3
train["warranty_texto"] = train["warranty"].fillna("")


# N
def medidas_fotos(lista_fotos):
    """Ancho y alto (en píxeles) de cada foto, a partir de max_size "ANCHOxALTO"."""
    medidas = []
    for foto in lista_fotos:
        try:
            ancho, alto = foto["max_size"].split("x")
            medidas.append((int(ancho), int(alto)))
        except (KeyError, AttributeError, ValueError):
            continue
    return medidas


def es_proporcion_camara(ancho, alto):
    return abs(ancho / alto - 4 / 3) < 0.01 or abs(ancho / alto - 3 / 4) < 0.01


medidas = train["pictures"].map(medidas_fotos)
train["foto_log_px_primera"] = medidas.map(lambda m: np.log(m[0][0] * m[0][1]) if m else np.nan)
train["foto_pct_cuadradas"] = medidas.map(lambda m: np.mean([a == h for a, h in m]) if m else np.nan)
train["foto_pct_camara"] = medidas.map(lambda m: np.mean([es_proporcion_camara(a, h) for a, h in m]) if m else np.nan)
train["foto_n_tamanios"] = medidas.map(lambda m: len(set(m)))

# L
train["last_updated"] = pd.to_datetime(train["last_updated"]).dt.tz_localize(None)
train["hora_inicio"] = train["start_time"].dt.hour
train["gap_actualizacion"] = (train["last_updated"] - train["start_time"]).dt.total_seconds() / 86400

columnas_numericas = columnas_numericas_base + [
    "foto_log_px_primera", "foto_pct_cuadradas", "foto_pct_camara", "foto_n_tamanios",
    "hora_inicio", "gap_actualizacion",
]

## Pipeline con caché del preprocesamiento

El preprocesamiento (capeo, logs, encoders, TF-IDF y sobre todo el modelo de texto de `PrediccionOOF`, que entrena 6 regresiones logísticas) **no depende de los hiperparámetros del HistGB**. Sin caché se recalcularía igual en cada una de las 30 combinaciones × 5 folds.

`Pipeline(memory=...)` (no está en el material) guarda en disco el resultado de ajustar cada paso de preprocesamiento sobre un conjunto de datos dado. Si después se pide **el mismo paso, con los mismos parámetros, sobre las mismas filas**, lo lee del disco en vez de recalcularlo. No cambia ningún resultado: solo evita repetir cuentas. Uso una carpeta temporal que se borra al final.

Para que comparar "las mismas filas" sea rápido, me quedo solo con las columnas que usa el modelo (`train_modelo`). Las columnas anidadas grandes (`pictures`, `attributes`, ...) ya se resumieron arriba.

In [3]:
columnas_texto = ["title", "warranty_texto"]
columnas_otras = ["price", "base_price", "currency_id", "available_quantity", "seller_id", "category_id"]
columnas_modelo = list(dict.fromkeys(columnas_categoricas_base + columnas_numericas + columnas_texto + columnas_otras))
columnas_modelo = [c for c in columnas_modelo if c not in ("log_price", "log_available_quantity")]
train_modelo = train[columnas_modelo]

carpeta_cache = tempfile.mkdtemp()
memoria = Memory(location=carpeta_cache, verbose=0)


def armar_modelo_texto_con_garantia(C):
    vectores = ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                       strip_accents="unicode"), "title"),
        ("garantia", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "warranty_texto"),
    ])
    return Pipeline([
        ("vectores", vectores),
        ("modelo", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
    ])


def armar_pipeline(modelo):
    """Configuración ganadora (T2a + N + L + T3) con el HistGB que se le pase."""
    preprocesador = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto_con_garantia(C=2), random_state=SEED), columnas_texto),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo),
    ], memory=memoria)

## Referencia: el modelo fijo

Tiene que dar 0,9339 (la configuración ganadora de `11b`). Esta corrida además **llena la caché**: cada fold queda preprocesado y las 30 combinaciones lo reusan.

In [4]:
modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    resultados_fijo = cross_validate(armar_pipeline(modelo_fijo), train_modelo, y, cv=cv,
                                     scoring=["accuracy", "roc_auc"])

accuracy_fijo = resultados_fijo["test_accuracy"]
print("Modelo fijo -> accuracy:", round(accuracy_fijo.mean(), 4), "+/-", round(accuracy_fijo.std(), 4),
      "| roc_auc:", round(resultados_fijo["test_roc_auc"].mean(), 4))

Modelo fijo -> accuracy: 0.9339 +/- 0.0018 | roc_auc: 0.9827


## Búsqueda: `RandomizedSearchCV` con 30 combinaciones

- `param_distributions`: la grilla (los nombres llevan `modelo__` porque el HistGB es el paso `"modelo"` del `Pipeline`).
- `scoring=["accuracy", "roc_auc"]`, `refit="accuracy"`: elige y reentrena la mejor por accuracy, pero guarda las dos.
- `n_jobs=4`: 4 ajustes en paralelo. No uso todos los núcleos porque cada HistGB ya usa varios hilos.

In [5]:
modelo_a_tunear = HistGradientBoostingClassifier(
    early_stopping=True, max_iter=2000, validation_fraction=0.1, n_iter_no_change=10, random_state=SEED
)

grilla = {
    "modelo__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "modelo__max_leaf_nodes": [15, 31, 63, 127],
    "modelo__min_samples_leaf": [10, 20, 50, 100],
    "modelo__l2_regularization": [0, 0.1, 1, 10],
}

busqueda = RandomizedSearchCV(
    armar_pipeline(modelo_a_tunear),
    param_distributions=grilla,
    n_iter=30,
    scoring=["accuracy", "roc_auc"],
    refit="accuracy",
    cv=cv,
    random_state=SEED,
    n_jobs=4,
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    busqueda.fit(train_modelo, y)

print("Mejor combinación:", busqueda.best_params_)
print("Accuracy CV:", round(busqueda.best_score_, 4))

Mejor combinación: {'modelo__min_samples_leaf': 10, 'modelo__max_leaf_nodes': 63, 'modelo__learning_rate': 0.03, 'modelo__l2_regularization': 1}
Accuracy CV: 0.9344


## Las 10 mejores combinaciones

In [6]:
resultados = pd.DataFrame(busqueda.cv_results_)
columnas_param = list(grilla.keys())
tabla_busqueda = resultados[["param_" + p for p in columnas_param] + ["mean_test_accuracy", "std_test_accuracy",
                                                                      "mean_test_roc_auc", "rank_test_accuracy"]].copy()
tabla_busqueda.columns = [c.replace("param_modelo__", "") for c in tabla_busqueda.columns]
tabla_busqueda.sort_values("rank_test_accuracy").head(10).round(4)

,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,mean_test_accuracy,std_test_accuracy,mean_test_roc_auc,rank_test_accuracy
14,0.03,63,10,1.0,0.9344,0.0022,0.9828,1
20,0.10,63,100,10.0,0.9342,0.0017,0.9828,2
12,0.10,31,50,10.0,0.9342,0.0019,0.9830,3
23,0.05,63,10,1.0,0.9341,0.0020,0.9828,4
8,0.03,63,20,0.0,0.9341,0.0019,0.9827,5
0,0.10,31,10,10.0,0.9340,0.0016,0.9830,6
28,0.03,63,50,0.0,0.9340,0.0019,0.9828,7
13,0.05,63,20,10.0,0.9340,0.0020,0.9829,8
19,0.05,63,10,0.0,0.9340,0.0018,0.9825,9
3,0.03,127,50,10.0,0.9339,0.0016,0.9828,10


## ¿Qué hiperparámetro importa más?

Promedio de accuracy de las 30 combinaciones según cada valor de cada hiperparámetro. Es una lectura gruesa (las combinaciones son al azar y no están balanceadas), pero muestra hacia dónde empuja cada uno.

In [7]:
for parametro in columnas_param:
    print(parametro.replace("modelo__", ""))
    print(resultados.groupby("param_" + parametro)["mean_test_accuracy"].agg(["mean", "max", "count"]).round(4))
    print()

learning_rate
                               mean     max  count
param_modelo__learning_rate                       
0.03                         0.9339  0.9344      9
0.05                         0.9336  0.9341      7
0.10                         0.9335  0.9342      7
0.20                         0.9329  0.9335      7

max_leaf_nodes
                                mean     max  count
param_modelo__max_leaf_nodes                       
15                            0.9329  0.9335      5
31                            0.9338  0.9342      9
63                            0.9338  0.9344      9
127                           0.9332  0.9339      7

min_samples_leaf
                                  mean     max  count
param_modelo__min_samples_leaf                       
10                              0.9333  0.9344     11
20                              0.9333  0.9341      6
50                              0.9338  0.9342      7
100                             0.9337  0.9342      6

l2_regula

## Mejor combinación vs modelo fijo, fold por fold

Misma comparación pareada que desde `10_vendedor_v2.ipynb`: mejora real si gana en 5 de 5 folds, o en 4 de 5 con la media de la diferencia mayor que su desvío.

**Limitación (sesgo de selección).** La mejor de 30 se eligió **mirando estos mismos folds**, así que su score está un poco inflado: entre 30 candidatas, alguna gana algo "de suerte". La solución rigurosa es una CV anidada (otra CV adentro de cada fold para elegir los hiperparámetros), pero multiplica el costo por 5. Por eso solo doy por buena una ganancia **clara**.

In [8]:
mejor = busqueda.best_index_
accuracy_mejor = np.array([resultados.loc[mejor, f"split{k}_test_accuracy"] for k in range(5)])
diferencias = accuracy_mejor - accuracy_fijo

print("Mejor tuneado -> accuracy:", round(accuracy_mejor.mean(), 4), "+/-", round(accuracy_mejor.std(), 4),
      "| roc_auc:", round(resultados.loc[mejor, "mean_test_roc_auc"], 4))
print("diferencia por fold:", np.round(diferencias, 4))
print("folds en que gana el tuneado:", (diferencias > 0).sum(), "de 5")
print("media de la diferencia:", round(diferencias.mean(), 4), "| desvío de la diferencia:", round(diferencias.std(), 4))
print()
print("Árboles que usó el mejor modelo (reentrenado con todo train):", busqueda.best_estimator_.named_steps["modelo"].n_iter_)

Mejor tuneado -> accuracy: 0.9344 +/- 0.0022 | roc_auc: 0.9828
diferencia por fold: [ 0.0012 -0.0005  0.      0.0001  0.0016]
folds en que gana el tuneado: 3 de 5
media de la diferencia: 0.0005 | desvío de la diferencia: 0.0008

Árboles que usó el mejor modelo (reentrenado con todo train): 342


In [9]:
shutil.rmtree(carpeta_cache, ignore_errors=True)

## Conclusión

| Modelo | Hiperparámetros | accuracy | roc_auc |
|---|---|---|---|
| Fijo | `learning_rate=0.1`, `max_iter=200`, resto por defecto (31 hojas, 20 por hoja, L2 = 0); paraba solo en ~115 árboles | 0,9339 ± 0,0018 | 0,9827 |
| Mejor de 30 | `learning_rate=0.03`, 63 hojas, 10 por hoja, L2 = 1; 342 árboles | 0,9344 ± 0,0022 | 0,9828 |

**Fold por fold:** el tuneado gana en 3 de 5 folds, con media +0,0005 y desvío 0,0008. **No pasa el criterio.** Además, ese +0,0005 es justo el tipo de ventaja que puede salir del sesgo de selección (la mejor de 30, elegida sobre los mismos folds).

**Lectura:**
- **La superficie es plana.** Las 10 mejores combinaciones están entre 0,9339 y 0,9344, y casi las 30 caen en un rango de ~0,0015. Para estos datos, el HistGB rinde parecido con casi cualquier configuración razonable. Lo que movió la aguja en este trabajo fueron **las features** (de 0,825 a 0,934), no los hiperparámetros.
- **Tendencias suaves**, coherentes con S7/S8:
  - `learning_rate` más chico es un poco mejor (0,03 > 0,05 > 0,1 > 0,2 en promedio). Con *early stopping*, el modelo compensa usando más árboles.
  - Árboles intermedios (31–63 hojas) son mejores que muy chicos (15, poco flexibles) o muy grandes (127, sobreajustan).
  - `min_samples_leaf` y `l2_regularization` casi no importan.
- **Decisión: mantener el modelo fijo.** Rinde igual dentro del ruido, es más simple de explicar y entrena ~3 veces más rápido (115 árboles contra 342).

**Para la presentación:** "tuneamos con `RandomizedSearchCV` (30 combinaciones, 5 folds) y la mejora no fue significativa: el modelo es robusto a sus hiperparámetros y el rendimiento lo explican las features".
